# Kandinsky Game→Audio — Step 10: Real Lite Checkpoint Pipeline Smoke Test

Steps 7–9 passed on tiny synthetic models/source checks. This is the first real-checkpoint compatibility run: load the official Kandinsky Lite Distill weights strictly into the patched model, attach a **zero-gated, untrained** Game→Audio branch, and pass a small event-token payload through a 5-second video→audio pipeline call.

The event tokens are deliberately placeholders; they have no agreed engine-independent meaning yet. The branch gate stays exactly zero, so this run is meant to preserve Kandinsky's baseline behavior while proving the real pipeline accepts the new interface.

Kandinsky still requires a text prompt in addition to the source video. This does **not** replace prompt conditioning.

Source-derived scratch patches are based on the MIT-licensed Kandinsky 6 code at the pinned revision. The upstream copyright and license notice is preserved in [THIRD_PARTY_NOTICES.md](THIRD_PARTY_NOTICES.md).

## License and resource notes

The official Lite Distill model card declares MIT; the pinned Kandinsky source repository also provides an MIT license. This supports a permissive prototype path, but it is not a complete commercial clearance: bundled third-party components, dependencies, training-data provenance, and output rights still need their own audit.

The pinned checkpoint snapshot contains about 26.64 GB of safetensors. Allow at least 30 GB of free disk. Use the A100 80 GB runtime and the tested module-offload + SDPA configuration; generation is 5 seconds and can still take several minutes. No API key or Hugging Face token is requested.

## What a pass means

- The patched PiFlow model strictly loads the real Lite Distill weights before new adapter modules are attached.
- The pipeline accepts `game_tokens=[1, events, 16]` and a boolean mask.
- The pipeline completes one video-conditioned audio generation and writes an MP4.
- All adapter gates remain zero; therefore this is compatibility plumbing, not learned game control or a sound-quality comparison.

If setup, download, or generation fails, report the first traceback and the preceding cell output. Do not retry the full download blindly if the disk-space check fails.

## 1. Check the Colab runtime

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise RuntimeError("Select an A100 GPU runtime before continuing.")
print(gpu.stdout.strip())
if "A100" not in gpu.stdout:
    raise RuntimeError("This smoke test is configured for an A100.")
disk = shutil.disk_usage("/content")
print(f"Free /content disk: {disk.free / 1e9:.1f} GB")
if disk.free < 30_000_000_000:
    raise RuntimeError("At least 30 GB free disk is needed for the pinned Lite Distill snapshot.")

## 2. Clone the pinned source and install its tested environment

We use a separate scratch checkout. The model config uses PyTorch SDPA. The official A100 setup also tries to compile SageAttention and requires `nvcc`; that optional step is skipped here.

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step10-game-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT], check=True)
    subprocess.run(["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT], check=True)
assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT
print("Pinned Kandinsky source:", K6_COMMIT)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# Match the official A100/CUDA dependency group but skip setup-gpu.sh's
# optional SageAttention source build. This notebook explicitly uses SDPA.
sync = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=K6_DIR, capture_output=True, text=True
)
print(sync.stdout[-6000:])
if sync.returncode:
    print(sync.stderr[-6000:])
    raise RuntimeError("CUDA dependency sync failed. The complete uv output is shown above.")

VENV_PYTHON = K6_DIR / ".venv/bin/python"
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f"Expected Kandinsky environment: {VENV_PYTHON}")

gpu_check = subprocess.run(
    [str(VENV_PYTHON), "-c",
     "import torch; print('Torch:', torch.__version__); "
     "print('CUDA available:', torch.cuda.is_available()); "
     "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); "
     "assert torch.cuda.is_available()"],
    cwd=K6_DIR, capture_output=True, text=True
)
print(gpu_check.stdout)
if gpu_check.returncode:
    print(gpu_check.stderr[-6000:])
    raise RuntimeError("The installed PyTorch environment cannot see the Colab GPU.")

## 3. Prepare exact pinned sources

In [ ]:
DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
EULER_PATH = K6_DIR / "kandinsky/core/algo/denoise_loop.py"
PIFLOW_PATH = K6_DIR / "kandinsky/core/algo/piflow_sampler.py"
PIPELINE_PATH = K6_DIR / "kandinsky/pipeline/pipeline.py"
PIFLOW_DIT_PATH = K6_DIR / "kandinsky/core/components/piflow_dit.py"
SAMPLER_CONFIG_PATH = K6_DIR / "kandinsky/runtime/sampler.py"
FACTORY_PATH = K6_DIR / "kandinsky/pipeline/factory.py"
WEIGHTS_PATH = K6_DIR / "kandinsky/runtime/weights.py"

originals = {
    "dit": DIT_PATH.read_text(encoding="utf-8"),
    "euler": EULER_PATH.read_text(encoding="utf-8"),
    "piflow": PIFLOW_PATH.read_text(encoding="utf-8"),
    "pipeline": PIPELINE_PATH.read_text(encoding="utf-8"),
    "piflow_dit": PIFLOW_DIT_PATH.read_text(encoding="utf-8"),
    "sampler_config": SAMPLER_CONFIG_PATH.read_text(encoding="utf-8"),
    "factory": FACTORY_PATH.read_text(encoding="utf-8"),
    "weights": WEIGHTS_PATH.read_text(encoding="utf-8"),
}
source = originals["dit"]

def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

print("Loaded pinned files:", ", ".join(originals))

## 4. Add the Game→Audio branch to a scratch DiT source copy

This is the full-DiT patch used in the earlier steps. The adapter is still absent during strict checkpoint loading.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

# Add the optional event-token width to the real fused block.
patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "fused-block constructor",
)

# Create the Game→Audio attention branch only when game_dim is configured.
patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "fused-block Game→Audio modules",
)

# Extend the fused block call. These are also used for audio-only calls.
patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "    ):\n",
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ):\n",
    "fused-block forward signature",
)

# Apply the branch after existing text/AV updates and before the audio FFN.
patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_tokens = game_tokens.to(\n"
    "                device=aud.device, dtype=self.game_to_audio.to_key.weight.dtype\n"
    "            )\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = game_attn_mask.to(device=aud.device, dtype=torch.bool)\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "fused-block Game→Audio insertion",
)

# Add game_dim to DiffusionTransformer3D and pass it to every fused block.
patched = replace_once(
    patched,
    "        visual_token_type_num_embeddings: int = 0,\n"
    "    ):\n",
    "        visual_token_type_num_embeddings: int = 0,\n"
    "        game_dim: int | None = None,\n"
    "    ):\n",
    "DiT constructor signature",
)
patched = replace_once(
    patched,
    "        self.is_multimodal  = is_multimodal\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "        self.is_multimodal  = is_multimodal\n"
    "        self.game_dim       = game_dim\n"
    "        if game_dim is not None and not is_multimodal:\n"
    "            raise ValueError(\"game_dim is supported only by the multimodal audio model.\")\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "DiT game_dim configuration",
)
patched = replace_once(
    patched,
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n",
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n"
    "                    game_dim=game_dim,\n",
    "pass game_dim to fused-block stack",
)

# Thread game tokens through the audio-only fused-block loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "single-modality block-loop signature",
)
patched = replace_once(
    patched,
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                    )\n",
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                        game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "                    )\n",
    "audio-only fused-block call",
)

# Thread the same tensors through the joint video+audio loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "fused block-loop signature",
)
patched = replace_once(
    patched,
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "joint fused-block call",
)

# Add public forward arguments and validate their basic contract.
patched = replace_once(
    patched,
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "public forward signature",
)
patched = replace_once(
    patched,
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens.\")\n"
    "        if game_tokens is not None:\n"
    "            if not self.is_multimodal or self.game_dim is None:\n"
    "                raise ValueError(\"Configure game_dim on a multimodal model before passing game_tokens.\")\n"
    "            if x_audio is None:\n"
    "                raise ValueError(\"Game conditioning requires x_audio.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim].\")\n"
    "            audio_batch = x_audio.shape[0] if x_audio.ndim == 3 else 1\n"
    "            if game_tokens.shape[0] != audio_batch:\n"
    "                raise ValueError(\"game_tokens and x_audio must have the same batch size.\")\n"
    "            if game_attn_mask is not None:\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means a valid event token.\")\n"
    "                if game_attn_mask.shape[-1] != game_tokens.shape[1]:\n"
    "                    raise ValueError(\"The final game_attn_mask dimension must match the event count.\")\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "forward argument validation",
)

# Pass the optional condition to audio-only and joint forward paths.
patched = replace_once(
    patched,
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "audio-only DiT call",
)
patched = replace_once(
    patched,
    "            vis_rope, aud_rope, attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "            vis_rope, aud_rope, attn_mask,\n"
    "            game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "joint DiT call",
)

PATCH_PATH = DIT_PATH.with_name("dit_game_full.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Patched scratch copy:", PATCH_PATH)
print("Original source unchanged:", DIT_PATH.read_text(encoding="utf-8") == source)

## 5. Make scratch PiFlow, Euler, and pipeline modules

In [ ]:
euler_patched = replace_once(
    originals["euler"],
    "    visual_token_type_ids: Tensor | None = None,\n"
    "    recompute_ropes_each_step: bool = False,\n",
    "    visual_token_type_ids: Tensor | None = None,\n"
    "    game_tokens: Tensor | None = None,\n"
    "    game_attn_mask: Tensor | None = None,\n"
    "    recompute_ropes_each_step: bool = False,\n",
    "Euler sampler condition arguments",
)
euler_patched = replace_once(
    euler_patched,
    "                visual_token_type_ids=visual_token_type_ids,\n"
    "            )\n\n"
    "        vel_cond = _forward(\n",
    "                visual_token_type_ids=visual_token_type_ids,\n"
    "                game_tokens=game_tokens,\n"
    "                game_attn_mask=game_attn_mask,\n"
    "            )\n\n"
    "        vel_cond = _forward(\n",
    "Euler DiT forwarding",
)

piflow_patched = replace_once(
    originals["piflow"],
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    game_tokens: torch.Tensor | None = None,\n"
    "    game_attn_mask: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "PiFlow sampler condition arguments",
)
piflow_patched = replace_once(
    piflow_patched,
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "            game_tokens=game_tokens,\n"
    "            game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "PiFlow DiT forwarding",
)

EULER_GAME_PATH = EULER_PATH.with_name("denoise_loop_game.py")
PIFLOW_GAME_PATH = PIFLOW_PATH.with_name("piflow_sampler_game.py")
EULER_GAME_PATH.write_text(euler_patched, encoding="utf-8")
PIFLOW_GAME_PATH.write_text(piflow_patched, encoding="utf-8")
print("Scratch Euler sampler:", EULER_GAME_PATH)
print("Scratch PiFlow sampler:", PIFLOW_GAME_PATH)

In [ ]:
print("Scratch Euler and PiFlow sampler modules are ready.")

The next cell applies the Step 9 public API patch to the pinned pipeline text, then writes it as a sibling module.

In [ ]:
pipeline_patched = originals["pipeline"]

pipeline_patched = replace_once(
    pipeline_patched,
    "from ..core.algo.denoise_loop import denoise_loop\n",
    "from ..core.algo.denoise_loop_game import denoise_loop\n",
    "pipeline Euler import",
)
pipeline_patched = replace_once(
    pipeline_patched,
    "from ..core.algo.piflow_sampler import piflow_denoise_loop\n",
    "from ..core.algo.piflow_sampler_game import piflow_denoise_loop\n",
    "pipeline PiFlow import",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "        audio: str | Path | None = None,\n"
    "        video: str | Path | None = None,\n"
    "    ) -> Kandinsky6PipelineOutput:\n",
    "        audio: str | Path | None = None,\n"
    "        video: str | Path | None = None,\n"
    "        game_tokens: torch.Tensor | None = None,\n"
    "        game_attn_mask: torch.Tensor | None = None,\n"
    "    ) -> Kandinsky6PipelineOutput:\n",
    "public pipeline event arguments",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "        generate_audio = do_audio and audio is None\n"
    "        generate_video = video is None\n",
    "        generate_audio = do_audio and audio is None\n"
    "        generate_video = video is None\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens\")\n"
    "        if game_tokens is not None:\n"
    "            game_tokens = torch.as_tensor(game_tokens)\n"
    "            if not generate_audio:\n"
    "                raise ValueError(\"game_tokens require generated audio output\")\n"
    "            if not getattr(dit, \"is_multimodal\", False):\n"
    "                raise ValueError(\"game_tokens require a multimodal audio model\")\n"
    "            game_dim = getattr(dit, \"game_dim\", None)\n"
    "            if game_dim is None:\n"
    "                raise ValueError(\"Attach a Game→Audio adapter before passing game_tokens\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[1] < 1 or game_tokens.shape[2] != game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim]\")\n"
    "            if not game_tokens.is_floating_point():\n"
    "                raise ValueError(\"game_tokens must contain floating-point features\")\n"
    "            if game_tokens.shape[0] != batch_size:\n"
    "                raise ValueError(\"game_tokens batch size must match the pipeline request\")\n"
    "            game_tokens = game_tokens.to(device=self.device)\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = torch.as_tensor(game_attn_mask)\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means valid event\")\n"
    "                if tuple(game_attn_mask.shape) != tuple(game_tokens.shape[:2]):\n"
    "                    raise ValueError(\"game_attn_mask must have shape [batch, events]\")\n"
    "                if not game_attn_mask.any(dim=1).all():\n"
    "                    raise ValueError(\"each request needs at least one valid event token\")\n"
    "                game_attn_mask = game_attn_mask[:, None, None, :].to(device=self.device)\n",
    "pipeline event validation and device placement",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "                        attention_mask=attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        scale_factor=self.scale_factor,\n",
    "                        attention_mask=attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        game_tokens=game_tokens,\n"
    "                        game_attn_mask=game_attn_mask,\n"
    "                        scale_factor=self.scale_factor,\n",
    "pipeline PiFlow condition call",
)
pipeline_patched = replace_once(
    pipeline_patched,
    "                        attention_mask=attention_mask,\n"
    "                        null_attention_mask=null_attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        scale_factor=self.scale_factor,\n",
    "                        attention_mask=attention_mask,\n"
    "                        null_attention_mask=null_attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        game_tokens=game_tokens,\n"
    "                        game_attn_mask=game_attn_mask,\n"
    "                        scale_factor=self.scale_factor,\n",
    "pipeline Euler condition call",
)

PIPELINE_GAME_PATH = PIPELINE_PATH.with_name("pipeline_game.py")
PIPELINE_GAME_PATH.write_text(pipeline_patched, encoding="utf-8")
print("Scratch pipeline written:", PIPELINE_GAME_PATH)
print("Original files still unchanged on disk:", all(
    path.read_text(encoding="utf-8") == originals[key]
    for key, path in {
        "dit": DIT_PATH,
        "euler": EULER_PATH,
        "piflow": PIFLOW_PATH,
        "pipeline": PIPELINE_PATH,
    }.items()
))

## 6. Route the factory to the patched model and pinned model snapshot

In [ ]:
piflow_dit_game = replace_once(
    originals["piflow_dit"],
    "from .dit import DiffusionTransformer3D\n",
    "from .dit_game_full import DiffusionTransformer3D\n",
    "PiFlow DiT base import",
)
(K6_DIR / "kandinsky/core/components/piflow_dit_game.py").write_text(piflow_dit_game, encoding="utf-8")

sampler_game = originals["sampler_config"]
sampler_game = replace_once(
    sampler_game,
    "from kandinsky.core.components.dit import DiffusionTransformer3D\n",
    "from kandinsky.core.components.dit_game_full import DiffusionTransformer3D\n",
    "sampler base DiT import",
)
sampler_game = replace_once(
    sampler_game,
    "from kandinsky.core.components.piflow_dit import PiFlowDiffusionTransformer3D\n",
    "from kandinsky.core.components.piflow_dit_game import PiFlowDiffusionTransformer3D\n",
    "sampler PiFlow DiT import",
)
(K6_DIR / "kandinsky/runtime/sampler_game.py").write_text(sampler_game, encoding="utf-8")

HF_REVISION = "4c487e8b4aeeb9059e5a77905aa2ddb59ec28f81"
weights_game = replace_once(
    originals["weights"],
    'CHECKPOINT_REVISION = "main"',
    f'CHECKPOINT_REVISION = "{HF_REVISION}"',
    "pin Lite Distill snapshot revision",
)
(K6_DIR / "kandinsky/runtime/weights_game.py").write_text(weights_game, encoding="utf-8")

factory_game = originals["factory"]
factory_game = replace_once(
    factory_game,
    "from ..core.components.dit import DiffusionTransformer3D\n",
    "from ..core.components.dit_game_full import DiffusionTransformer3D\n",
    "factory DiT import",
)
factory_game = replace_once(
    factory_game,
    "from ..runtime.sampler import dit_class, reject_incompatible_cache\n",
    "from ..runtime.sampler_game import dit_class, reject_incompatible_cache\n",
    "factory sampler import",
)
factory_game = replace_once(
    factory_game,
    "from ..runtime.weights import ensure_checkpoint, load_bf16_checkpoint, materialize_meta_buffers\n",
    "from ..runtime.weights_game import ensure_checkpoint, load_bf16_checkpoint, materialize_meta_buffers\n",
    "factory pinned weights import",
)
factory_game = replace_once(
    factory_game,
    "from .pipeline import Kandinsky6Pipeline\n",
    "from .pipeline_game import Kandinsky6Pipeline\n",
    "factory pipeline import",
)
(K6_DIR / "kandinsky/pipeline/factory_game.py").write_text(factory_game, encoding="utf-8")

print("Scratch modules written. Official source files remain unchanged.")
print("Pinned HF model snapshot:", HF_REVISION)

## 7. Write the small inference config

In [ ]:
CONFIG_PATH = K6_DIR / "kandinsky/configs/devices/a100-lite-distill-game-smoke.yaml"
CONFIG_PATH.write_text(
    "checkpoint: lite-distill\n"
    "cache:\n  mode: none\n"
    "offload:\n  strategy: module\n"
    "compile:\n  strategy: none\n"
    "attention:\n  engine: sdpa\n"
    "text_embedder:\n  quantized_qwen: false\n"
    "beautifier:\n  name: qwen25\n",
    encoding="utf-8",
)
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 8. Provide a five-second source video

The source video is the visual condition. Kandinsky will generate only its audio. If the earlier file is not present in this runtime, upload an MP4 of at least five seconds.

In [ ]:
from google.colab import files

SOURCE_VIDEO = Path("/content/kandinsky_v2a_input.mp4")
if not SOURCE_VIDEO.exists():
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Upload one MP4 source video.")
    name, data = next(iter(uploaded.items()))
    SOURCE_VIDEO.write_bytes(data)

probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(SOURCE_VIDEO)
], capture_output=True, text=True, check=True)
source_seconds = float(probe.stdout.strip())
if source_seconds < 5:
    raise ValueError(f"Source video is shorter than 5 seconds: {source_seconds:.2f}s")

INPUT_VIDEO = Path("/content/game_conditioned_source_5s.mp4")
prepared = subprocess.run([
    "ffmpeg", "-y", "-i", str(SOURCE_VIDEO), "-t", "5",
    "-map", "0:v:0", "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-r", "24", str(INPUT_VIDEO)
], capture_output=True, text=True)
if prepared.returncode:
    raise RuntimeError(prepared.stderr[-2000:])
print("Five-second video ready:", INPUT_VIDEO)

## 9. Run one real-checkpoint pipeline call

The script loads the pinned Lite Distill snapshot through the normal strict loader. Only after that load succeeds, it attaches a tiny zero-gated branch. The placeholder event values are passed through the same API but are intentionally not learned or interpreted.

In [ ]:
OUTPUT_MP4 = Path("/content/kandinsky_game_api_smoke.mp4")
RUN_SCRIPT = K6_DIR / "run_game_api_smoke.py"
RUN_SCRIPT.write_text(r'''
import json
import sys
from pathlib import Path

import torch
from kandinsky.core.components.dit_game_full import MultiheadCrossAttention, Modulation
from kandinsky.pipeline.factory_game import get_pipeline

config_path = Path(sys.argv[1])
video_path = Path(sys.argv[2])
output_path = Path(sys.argv[3])
game_dim = 16

print("Loading pinned Lite Distill checkpoint and scratch pipeline...")
pipe = get_pipeline(str(config_path), device="cuda", attention_engine="sdpa", cache_mode="none")
model = pipe.raw_dit
if not getattr(model, "is_multimodal", False):
    raise RuntimeError("Loaded checkpoint is not a multimodal audio/video DiT.")
if getattr(model, "game_dim", None) is not None:
    raise RuntimeError("Expected an unadapted base model before sidecar attachment.")

for block in model.visual_transformer_blocks:
    reference = next(block.parameters())
    device, dtype = reference.device, reference.dtype
    head_dim = block.av_cross_attention.query_norm.normalized_shape[0]
    time_dim_a = block.audio_dec_block.visual_modulation.out_layer.in_features

    block.game_dim = game_dim
    block.game_audio_norm = torch.nn.LayerNorm(
        block.model_dim_a, elementwise_affine=False, device=device, dtype=dtype
    )
    block.game_to_audio = MultiheadCrossAttention(
        block.model_dim_a, head_dim, game_dim, "sdpa"
    ).to(device=device, dtype=dtype)
    block.game_audio_modulation = Modulation(
        time_dim_a, block.model_dim_a, 1
    ).to(device=device, dtype=dtype)
    # Keep the new branch exactly closed; this is a compatibility check, not training.
    torch.nn.init.zeros_(block.game_audio_modulation.out_layer.weight)
    torch.nn.init.zeros_(block.game_audio_modulation.out_layer.bias)

model.game_dim = game_dim
gate_max = max(
    block.game_audio_modulation.out_layer.weight.detach().abs().max().item()
    for block in model.visual_transformer_blocks
)
gate_bias_max = max(
    block.game_audio_modulation.out_layer.bias.detach().abs().max().item()
    for block in model.visual_transformer_blocks
)
assert gate_max == 0 and gate_bias_max == 0
print("Real base checkpoint strict-loaded before adapter attachment: passed")
print("Attached zero-gated branches:", len(model.visual_transformer_blocks))

torch.manual_seed(42)
game_tokens = torch.randn(1, 3, game_dim, dtype=torch.float32)
game_attn_mask = torch.tensor([[True, True, False]], dtype=torch.bool)
prompt = "A single wooden block drops onto a hard wooden floor with one short hollow thud. No music or speech."

result = pipe(
    text=prompt,
    video=str(video_path),
    seed=42,
    num_steps=10,
    guidance_weight=1.0,
    sample_audio=True,
    save_path=str(output_path),
    game_tokens=game_tokens,
    game_attn_mask=game_attn_mask,
)
if not output_path.is_file():
    raise FileNotFoundError(f"Pipeline returned without saving {output_path}")

report = {
    "experiment": "step10_real_lite_game_api_smoke",
    "checkpoint_repo": "kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers",
    "checkpoint_revision": "4c487e8b4aeeb9059e5a77905aa2ddb59ec28f81",
    "source_commit": "7ceeefad7fb625011f7db602c43893a989f19c15",
    "game_dim": game_dim,
    "game_tokens_shape": list(game_tokens.shape),
    "game_mask_shape": list(game_attn_mask.shape),
    "game_branch_gate_zero": gate_max == 0 and gate_bias_max == 0,
    "output_mp4": str(output_path),
    "output_bytes": output_path.stat().st_size,
    "generation_completed": True,
    "adapter_trained": False,
    "event_schema_semantic": False,
}
report_path = output_path.with_suffix(".json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
''', encoding="utf-8")

run = subprocess.run(
    [str(VENV_PYTHON), str(RUN_SCRIPT), str(CONFIG_PATH), str(INPUT_VIDEO), str(OUTPUT_MP4)],
    cwd=K6_DIR, capture_output=True, text=True
)
print(run.stdout[-8000:])
if run.returncode:
    print(run.stderr[-8000:])
    raise RuntimeError("Real-checkpoint pipeline smoke test failed.")
print("Generated:", OUTPUT_MP4)

## 10. Listen to the result and save the report

In [ ]:
from IPython.display import Audio, Video, display
import json

display(Video(str(OUTPUT_MP4), embed=True))
report = json.loads(OUTPUT_MP4.with_suffix(".json").read_text(encoding="utf-8"))
print("Gate stays closed:", report["game_branch_gate_zero"])
print("This confirms real-checkpoint API compatibility only; the event payload was not learned.")

## Interpretation and next step

A pass means the pinned Lite Distill checkpoint loads strictly into the expanded architecture and the modified public pipeline completes a real video-conditioned audio call with a game-token payload. Because the adapter gate is zero, the event vector is intentionally ignored; expect baseline Kandinsky behavior.

Do not call this game control, fine-tuning, or improved sound. The next meaningful experiment is to define a canonical event representation and obtain paired event/audio training examples. Only then can we train the adapter and compare closed-gate baseline against learned event control.